# End-to-End Cluster Execution Flow: From Code to Tasks

**Module**: 01_spark_fundamentals  
**Notebook**: 04_spark_cluster_execution.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The step-by-step lifecycle of an Apache Spark job: from code declaration to DAG construction, stage splitting, task scheduling, and result return.
- **Why does this topic exist?**: To diagnose bottlenecks, you must be able to trace exactly where a query is spending time: DAGScheduler, TaskScheduler, or network shuffle.
- **What problem does it solve?**: Without understanding execution hierarchy (Action -> Job -> Stage -> Task), developers cannot decipher the Spark UI.
- **Where is it used in real-world Data Engineering?**: Explaining execution flow and debugging stuck stages are top-tier Senior Data Engineering interview topics.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Spark architecture, SparkSession |
| **Required depth** | Deep working knowledge: can trace an action from code to tasks |
| **Data Engineer** | Explain jobs, stages, and tasks and use the Spark UI to follow an action. |
| **Senior Data Engineer** | Diagnose stage boundaries, retries, and locality from the execution graph. |
| **Spark specialist** | Reason about DAGScheduler decisions, shuffle map outputs, and task set scheduling. |
| **Next topic** | `02_rdd/01_rdd_fundamentals.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Code that looks like one operation can produce multiple jobs, stages, and thousands of tasks. Without a mental
model of the execution flow, a slow job is a mystery and a failed task is indistinguishable from a failed stage.
This notebook connects the user-facing API to the scheduler's actual units of work.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Job** | One action's execution | Submitted to the DAGScheduler by an action |
| **Stage** | A group of tasks separated by shuffles | ShuffleMapStage or ResultStage |
| **Task** | One partition's work | ShuffleMapTask or ResultTask |
| **Partition** | A chunk of distributed data | One task per partition per stage |
| **Shuffle** | Redistribution of data | Wide dependency and stage boundary |
| **Locality** | Where a task runs relative to data | PROCESS_LOCAL, NODE_LOCAL, RACK_LOCAL, ANY |

### How it works, step by step

1. The action calls `sc.runJob()`.
2. The DAGScheduler walks the RDD/plan graph backward.
3. Shuffle dependencies split the graph into stages.
4. Each stage becomes a TaskSet with one task per partition.
5. The TaskScheduler offers tasks to executors, respecting locality and resources.
6. The driver tracks task completion and retries failures.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Complete Execution Hierarchy

```
[User Action: df.count()]
        |
        v
+-----------------------------------------------------------+
| JOB (1 per Action)                                        |
|  Identified by Job ID (e.g. Job 0, Job 1)                 |
+-----------------------------+-----------------------------+
                              |
                              v DAGScheduler splits at Shuffles
+-----------------------------------------------------------+
| STAGES (Bounded by Wide Dependencies)                     |
|  - Stage 0: ShuffleMapStage (reads, filters, map output)  |
|  - Stage 1: ResultStage (fetches shuffle, aggregates)     |
+-----------------------------+-----------------------------+
                              |
                              v TaskScheduler assigns to Cores
+-----------------------------------------------------------+
| TASKS (1 per Partition in Stage)                          |
|  - Task 0 -> Core 1 on Worker 1                           |
|  - Task 1 -> Core 2 on Worker 1                           |
|  - Task 2 -> Core 1 on Worker 2                           |
|  - Task 3 -> Core 2 on Worker 2                           |
+-----------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Execution_Flow_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Generating a Multi-Stage Job

Let us run an operation with an explicit wide dependency (shuffle) and trace its job and stage breakdown.

In [ ]:
orders_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# A 2-stage execution: Filter (narrow) + GroupBy (wide)
agg_df = (
    orders_df
    .filter("status = 'Completed'")
    .groupBy("channel")
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_revenue")
    )
)

print("=== Execution Plan ===")
agg_df.explain("formatted")

## 4. Triggering the Action and Checking the Spark UI

Now we trigger the Action (`show()`) and inspect the generated Job in the Spark UI (`http://localhost:4040`).

In [ ]:
start_time = time.time()
agg_df.show()
elapsed = time.time() - start_time
print(f"Action executed in: {elapsed:.3f} seconds")
print("Check http://localhost:4040/jobs/ to inspect Job, Stages, and Tasks metrics!")

---

## Experiment: Predicting and Verifying Stage Count

- **1. What are we testing?**: Verify that a query with 2 wide transformations (groupBy + repartition) produces exactly 3 stages.
- **2. What do we expect?**: The DAG will have 3 stages separated by 2 shuffle exchange boundaries.

In [ ]:
# Query with 2 shuffles: groupBy followed by repartition
multi_shuffle = (
    orders_df
    .groupBy("customer_id")
    .count()
    .repartition(4)
)

# Observe Exchange nodes in plan
multi_shuffle.explain()
print(f"Result count: {multi_shuffle.count()}")

**3. What happened?**: Spark UI shows 3 stages for this job (Scan & partial group, shuffle merge & recount, repartition exchange).  
**4. Why did it happen?**: Each wide dependency (Exchange node) creates a new Stage boundary.  
**5. Practical Engineering Lesson**: Counting Exchange nodes in explain() allows you to predict the exact number of stages in the Spark UI.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: SLA investigation for a nightly pipeline

- **Industry**: Banking
- **Scenario**: A nightly reconciliation job misses its 6 a.m. deadline even though the cluster has free capacity.
- **Problem**: The job has more stages than expected, and one shuffle stage dominates the runtime.
- **Why this concept is relevant**: Tracing jobs → stages → tasks reveals where the time goes and whether the bottleneck is a shuffle, a straggler, or task launch overhead.
- **Implementation idea**: Open the Stages tab, compare shuffle read/write and task duration percentiles, then inspect the SQL plan for unnecessary exchanges.
- **Expected result**: A prioritized list of optimizations instead of guessing at cluster size.
- **Alternative**: A profiler or query-plan analyzer in a warehouse.
- **Trade-offs**: Spark UI gives distributed task-level evidence; a warehouse profiler gives SQL-level evidence for warehouse workloads.

### Use Case 2: Onboarding through a controlled pipeline

- **Industry**: Education / training
- **Scenario**: A new engineer needs to see how a simple `groupBy` becomes a distributed job.
- **Problem**: The concept remains abstract until the learner maps each code line to a UI stage.
- **Why this concept is relevant**: A small, deterministic pipeline with one shuffle makes the job/stage/task hierarchy visible.
- **Implementation idea**: Run the pipeline, predict the number of stages, then verify in the UI.
- **Expected result**: A correct prediction and an explanation of each stage boundary.
- **Alternative**: A local Spark UI demo or a recorded walkthrough.
- **Trade-offs**: The lab is interactive and uses real executors; a recording is cheaper but passive.

---

## Business Value

- Faster root-cause analysis for missed SLAs.
- Better capacity decisions based on actual stage behavior.
- Clearer communication between engineers and platform teams.
- Reduced guesswork in performance tuning.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use the Spark UI as the source of truth, not intuition.
- Track stage retries and failed tasks; a successful job can still be unstable.
- Watch for skew: max task duration far above the 75th percentile.
- Avoid collecting large results to the driver at the end of a job.
- Persist event logs for post-mortem analysis.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Predict stages before running

**Steps**

1. Build a query with a filter, a `groupBy`, and an `orderBy`.
2. Write down the expected number of stages and where the shuffles occur.
3. Run the query and compare with the Stages tab.
4. Inspect the SQL tab for Exchange nodes.

**What to observe**: Each shuffle boundary creates a new stage; AQE may coalesce partitions but does not remove the boundary.

**Healthy result**: Your prediction matches the UI and you can explain each boundary.

**Problematic result**: Unexpected stages usually mean an implicit shuffle or a reused exchange.

**Engineering takeaway**: Read the plan and the UI together.

---

## Trade-offs and Alternatives

### Advantages

- Clear unit of scheduling and fault isolation
- Enables locality-aware execution
- Makes shuffle costs visible

### Limitations

- Stage boundaries are not always obvious from code
- AQE can change the physical plan at runtime
- Task-level metrics require interpretation

### When to use it

- Debugging slow jobs
- Capacity planning
- Teaching distributed execution

### When not to use it

- Optimizing without measuring
- Assuming one action equals one stage

### Alternatives inside Spark

- DAGScheduler
- TaskScheduler
- Spark UI

### Alternatives outside Spark

- Distributed tracing
- Cluster-level monitoring
- Query profilers

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Narrow Transformation Chain
**Problem Statement**: Write a pipeline chaining filter, withColumn, and select. How many stages will count() produce?  
**Input / Context**: orders_df

In [ ]:
# Exercise 1: Count stages for narrow pipeline

### Exercise 2 (Intermediate): Trace Task Count from Partition Count
**Problem Statement**: Set shuffle partitions to 5, execute a groupBy count, and check the task count in the final stage.  
**Input / Context**: spark.conf.set('spark.sql.shuffle.partitions','5')

In [ ]:
# Exercise 2: Tune and verify task count

### Exercise 3 (Advanced): End-to-End Latency Breakdown
**Problem Statement**: Measure wall-clock latency of reading 60k orders, performing aggregation, and writing to Parquet.  
**Input / Context**: retail/orders.csv

In [ ]:
# Exercise 3: Full pipeline timing

---

## Production Debugging Exercise

### Too many stages

**Scenario**: A pipeline with five `repartition()` calls creates many stages and runs slowly.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether each repartition is necessary.
2. Look for `Exchange` nodes in the plan.
3. Consider `coalesce` for reducing partitions without a shuffle.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Narrow Transformation Chain):
```python
narrow_df = orders_df.filter("quantity > 1").withColumn("tax", F.col("unit_price")*0.1).select("order_id","tax")
# Has ZERO Exchange nodes -> exactly 1 Stage!
narrow_df.explain()
print("Count:", narrow_df.count())
```

#### Solution 2 (Trace Task Count from Partition Count):
```python
spark.conf.set("spark.sql.shuffle.partitions","5")
res = orders_df.groupBy("status").count()
print("Number of partitions in aggregated DF:", res.rdd.getNumPartitions())
res.show()
spark.conf.set("spark.sql.shuffle.partitions","8")
```

#### Solution 3 (End-to-End Latency Breakdown):
```python
out_path ="/opt/spark-data/warehouse/test_perf_out"
t0 = time.time()
(
    orders_df
    .filter("status ='Completed'")
    .groupBy("product_id")
    .agg(F.sum("quantity").alias("total_qty"))
    .coalesce(1)
    .write.mode("overwrite").parquet(out_path)
)
print(f"Full pipeline executed and written to Parquet in {time.time()-t0:.2f}s")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Hierarchy**: Action triggers Job -> Job splits into Stages at shuffles -> Stages split into Tasks per partition.
- **Pipelining**: Narrow operations execute together in a single stage without intermediate disk or network hops.
- **Task Execution**: Each task runs on exactly one partition on an executor thread.

### Essential Technical Terms
- `Job`: Execution triggered by one Action.
- `Stage`: Pipelined set of tasks separated by shuffle dependencies.
- `Task`: Individual thread unit processing a single partition.

### Common Engineering Mistakes
- **Expecting Stages for Every Operation**: Only wide transformations (shuffles) create stage boundaries; narrow operations are fused.
- **Ignoring Task Count**: Too few tasks leaves cores idle; too many tasks floods scheduler.

### Interview Questions on This Topic
- **Q: What determines the number of tasks in a stage?**
  - *A*: The number of partitions in the RDD/DataFrame entering that stage.
- **Q: What causes a new stage to be created?**
  - *A*: A shuffle dependency (wide transformation) such as groupBy, join, or repartition.




### More Interview Questions

- **Q: What is the relationship between a job, a stage, and a task?**
  - *A*: An action creates a job; the DAGScheduler splits the job into stages at shuffle boundaries; each stage contains one task per partition.
- **Q: Why does a `groupBy` create a stage boundary?**
  - *A*: It requires a wide dependency: data with the same key must be brought to the same partition, which requires a shuffle.

### What to Study Next
Module 01 complete! Proceed to Module 02: Resilient Distributed Datasets (RDDs).

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')